![HydroCycle](images/hydro_5cycle.jpg)

# Retrieve and Analyze Hydrology data for a watershed of interest

To make predictions for reservoir operations, water supply, flood control, etc, we need to collect data to train/calibrate hydrologic models. This includes streamflow, current environmental conditions (e.g., snow water equivalent), and future weather predictions. This exercise will build on the previous SNOTEL module to work towards building a hydrologic module.

Need to find a station? Use the [USGS NWIS mapper system](https://apps.usgs.gov/nwismapper/)


Click the link and explore!

## Create a HydroDataFrame

We want to create one data frame containing streamflow, meteological information, and SNOTEL for our period of record

In [21]:
from pynhd import NLDI
import geopandas as gpd
import pandas as pd
from supporting_scripts import getData, SNOTEL_Analyzer, dataprocessing, mapping
from shapely.geometry import box, Polygon
import os
import datetime
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [22]:
station_id = "11274790" # NWIS id for Tuolumne river at the mouth of Hetch Hetchy Reservoir
basinname = 'TuolumneRiverBasin'

## Load data

We need to load the saved data into our script to process and combine.

We will start with the SNOTEL data

In [23]:
#load snotel data
unprocessed_SNOTEL = {}
#read all files in the following path into the dictionary
path = 'files/SNOTEL'
for filename in os.listdir(path):
    if filename.endswith('.csv'):
        #select the name of the file between the _ and _
        name = filename.split('_')[1] 
        unprocessed_SNOTEL[name] = pd.read_csv(os.path.join(path, filename))
        #make the date a datetime object and set to the index
        unprocessed_SNOTEL[name]['Date'] = pd.to_datetime(unprocessed_SNOTEL[name]['Date'])
        unprocessed_SNOTEL[name].set_index('Date', inplace=True)
        #rename the Snow Water Equivalent (m) Start of Day Values to SWE_cm
        unprocessed_SNOTEL[name].rename(columns={'Snow Water Equivalent (m) Start of Day Values': f"{name}_SWE_cm"}, inplace=True)
        #convert SWE_m to cm
        unprocessed_SNOTEL[name][f"{name}_SWE_cm"] = unprocessed_SNOTEL[name][f"{name}_SWE_cm"] * 100
        #remove the Water_Year column
        unprocessed_SNOTEL[name].drop(columns=['Water_Year'], inplace=True)
        #we need to know how many obs for each DF, print the df name, its length, and the start/end dates
        print(f"{name}: {len(unprocessed_SNOTEL[name])} start date: {unprocessed_SNOTEL[name].index.min()} end date: {unprocessed_SNOTEL[name].index.max()}")
    



1013: 155 start date: 2025-10-01 00:00:00 end date: 2026-03-04 00:00:00
1013: 155 start date: 2025-10-01 00:00:00 end date: 2026-03-04 00:00:00
1113: 155 start date: 2025-10-01 00:00:00 end date: 2026-03-04 00:00:00
1113: 155 start date: 2025-10-01 00:00:00 end date: 2026-03-04 00:00:00
1115: 155 start date: 2025-10-01 00:00:00 end date: 2026-03-04 00:00:00
1115: 155 start date: 2025-10-01 00:00:00 end date: 2026-03-04 00:00:00
823: 155 start date: 2025-10-01 00:00:00 end date: 2026-03-04 00:00:00
823: 155 start date: 2025-10-01 00:00:00 end date: 2026-03-04 00:00:00
DAN: 6195 start date: 2004-10-01 00:00:00 end date: 2021-09-16 00:00:00
SLI: 6257 start date: 2004-10-01 00:00:00 end date: 2021-11-17 00:00:00
TES: 629 start date: 2005-03-01 00:00:00 end date: 2006-11-19 00:00:00
TUM: 6257 start date: 2004-10-01 00:00:00 end date: 2021-11-17 00:00:00


In [24]:
#The TES site is missing many values and will not be useful for our analysis, remove it
unprocessed_SNOTEL.pop('TES', None)

#The site with the latest start date will guide the rest
latest_start_date = max([df.index.min() for df in unprocessed_SNOTEL.values()])

#The site with the earliest end date will guide the rest
soonest_end_date = min([df.index.max() for df in unprocessed_SNOTEL.values()])
for key in unprocessed_SNOTEL.keys():
    unprocessed_SNOTEL[key] = unprocessed_SNOTEL[key][unprocessed_SNOTEL[key].index >= latest_start_date]
    unprocessed_SNOTEL[key] = unprocessed_SNOTEL[key][unprocessed_SNOTEL[key].index <= soonest_end_date]

#merge all dictionary dataframes into one larger dataframe
SNOTEL_df = pd.concat(unprocessed_SNOTEL.values(), axis=1)
#set the date index to be the index of the first dataframe in the dictionary

SNOTEL_df.head()

,1013_SWE_cm,1113_SWE_cm,1115_SWE_cm,823_SWE_cm,DAN_SWE_cm,SLI_SWE_cm,TUM_SWE_cm
Date,,,,,,,


## Load the meteorologogical data from Daymet and NLDAS

In [25]:
#Read the data from PyDayMet 
PyDayMet_df = pd.read_csv(f"files/PyDayMet/PyDayMet_{station_id}.csv")
#set the date column to be a datetime object and set it to the index
PyDayMet_df['Date'] = pd.to_datetime(PyDayMet_df['Date'])
PyDayMet_df.set_index('Date', inplace=True)
PyDayMet_df.head()

,dayl_s,prcp_mm_day,srad_W_m2,swe_cm,tmax_C,tmin_C,vp_Pa,tmean
Date,,,,,,,,
1980-01-01,33914.08,0.0,356.43,2.943,6.07,-12.54,232.94,-3.235
1980-01-02,33952.03,0.0,363.52,2.943,7.82,-11.91,239.32,-2.045
1980-01-03,33993.09,0.0,364.49,2.943,7.14,-12.50,229.62,-2.680
1980-01-04,34037.22,0.0,340.58,2.943,5.89,-10.41,276.75,-2.260
1980-01-05,34084.41,0.0,318.39,2.943,6.51,-7.77,340.48,-0.630


In [26]:
#Read the data from NLDAS 
NLDAS_df = pd.read_csv(f"files/NLDAS/NLDAS_{station_id}.csv")
#set the date column to be a datetime object and set it to the index
NLDAS_df['Date'] = pd.to_datetime(NLDAS_df['Date'])
NLDAS_df.set_index('Date', inplace=True)
NLDAS_df.head()

,convective_fraction,longwave_radiation,potential_energy,potential_evaporation,pressure,shortwave_radiation,specific_humidity,temperature,total_precipitation,wind_u,wind_v
Date,,,,,,,,,,,
2006-01-01,0.005192,199.833836,1.508087,0.044990,71408.170160,95.234192,0.002439,-7.561317,0.563703,2.807882,5.710911
2006-01-02,0.051767,264.132482,33.055034,0.013663,70799.285724,93.932771,0.004772,-0.944478,4.532128,4.467674,8.864378
2006-01-03,0.016707,214.292827,9.209950,0.030093,71724.799553,101.167170,0.002966,-6.371842,0.575387,4.247262,3.778855
2006-01-04,0.000000,226.581243,0.000000,0.016272,72808.304958,101.368610,0.003240,-3.462835,0.002358,1.485389,1.450410
2006-01-05,0.000000,214.910143,0.000000,0.033974,73429.441947,112.546583,0.002849,1.320337,0.000000,-0.779809,3.417216


## Load the streamflow data

In [27]:
from supporting_scripts.getData import get_usgs_streamflow

streamflow_df = get_usgs_streamflow(station_id)

streamflow_df.index = pd.to_datetime(streamflow_df.index)
streamflow_df.rename(columns={"Streamflow_cfs": "flow_cms"}, inplace=True)

Retrieving data for Site: 11274790 from 1980-01-01 to 2026-03-26...


In [37]:
# FIX timezone issue FIRST
SNOTEL_df.index = pd.to_datetime(SNOTEL_df.index).tz_localize(None)
PyDayMet_df.index = pd.to_datetime(PyDayMet_df.index).tz_localize(None)
NLDAS_df.index = pd.to_datetime(NLDAS_df.index).tz_localize(None)
streamflow_df.index = pd.to_datetime(streamflow_df.index).tz_localize(None)

# merge without clipping
Hydro_df = pd.concat([SNOTEL_df, PyDayMet_df, NLDAS_df, streamflow_df], axis=1)

# fill missing values
Hydro_df = Hydro_df.fillna(0)

Hydro_df.head()

,1013_SWE_cm,1113_SWE_cm,1115_SWE_cm,823_SWE_cm,DAN_SWE_cm,SLI_SWE_cm,TUM_SWE_cm,dayl_s,prcp_mm_day,srad_W_m2,...,pressure,shortwave_radiation,specific_humidity,temperature,total_precipitation,wind_u,wind_v,site_no,00060_Mean,00060_Mean_cd


## Load our catchment information

In [29]:
# basin_info = pd.read_csv(f"files/basin_info/basin_info_{station_id}.csv")
# basin_info.head()

## Now we can merge into one dataframe

In [39]:
# FIX timezone
SNOTEL_df.index = pd.to_datetime(SNOTEL_df.index).tz_localize(None)
PyDayMet_df.index = pd.to_datetime(PyDayMet_df.index).tz_localize(None)
NLDAS_df.index = pd.to_datetime(NLDAS_df.index).tz_localize(None)
streamflow_df.index = pd.to_datetime(streamflow_df.index).tz_localize(None)

# ONLY keep good SNOTEL stations (drop the new broken ones)
SNOTEL_df = SNOTEL_df[["TUM_SWE_cm", "DAN_SWE_cm", "SLI_SWE_cm"]]

# merge everything
Hydro_df = pd.concat([SNOTEL_df, PyDayMet_df, NLDAS_df, streamflow_df], axis=1)

# fill missing
Hydro_df = Hydro_df.fillna(0)

Hydro_df.head()

,TUM_SWE_cm,DAN_SWE_cm,SLI_SWE_cm,dayl_s,prcp_mm_day,srad_W_m2,swe_cm,tmax_C,tmin_C,vp_Pa,...,pressure,shortwave_radiation,specific_humidity,temperature,total_precipitation,wind_u,wind_v,site_no,00060_Mean,00060_Mean_cd


In [40]:
df = Hydro_df.copy()
df = df[(df.index >= "2018-10-01") & (df.index <= "2019-09-30")]
df.head()

,TUM_SWE_cm,DAN_SWE_cm,SLI_SWE_cm,dayl_s,prcp_mm_day,srad_W_m2,swe_cm,tmax_C,tmin_C,vp_Pa,...,pressure,shortwave_radiation,specific_humidity,temperature,total_precipitation,wind_u,wind_v,site_no,00060_Mean,00060_Mean_cd


In [31]:
#merge the SNOTEL_df, met_df, and streamflow dataframes
Hydro_df = pd.concat([SNOTEL_df, PyDayMet_df, NLDAS_df,streamflow_df], axis=1)
#put the site_no column, second to last, and streamfow column, last column, as the first two columns in the dataframe
cols = Hydro_df.columns.tolist()
cols = cols[-2:] + cols[:-2]
Hydro_df = Hydro_df[cols]
Hydro_df.head()

,00060_Mean,00060_Mean_cd,1013_SWE_cm,1113_SWE_cm,1115_SWE_cm,823_SWE_cm,DAN_SWE_cm,SLI_SWE_cm,TUM_SWE_cm,dayl_s,...,potential_energy,potential_evaporation,pressure,shortwave_radiation,specific_humidity,temperature,total_precipitation,wind_u,wind_v,site_no


In [32]:
#all of the NaN values here should be 0, fill them
Hydro_df = Hydro_df.fillna(0)
Hydro_df.head()

,00060_Mean,00060_Mean_cd,1013_SWE_cm,1113_SWE_cm,1115_SWE_cm,823_SWE_cm,DAN_SWE_cm,SLI_SWE_cm,TUM_SWE_cm,dayl_s,...,potential_energy,potential_evaporation,pressure,shortwave_radiation,specific_humidity,temperature,total_precipitation,wind_u,wind_v,site_no


In [33]:
# #add in the basin info as columns in the dataframe, repeat the values for each row
# for col in basin_info.columns:
#     Hydro_df[col] = basin_info[col][0]

# Hydro_df.head()

In [34]:
#take a look around peak SWE to make sure we have snotel values, early season can be tricky to assess
Hydro_df.loc['2019-03-01':'2019-04-01']

,00060_Mean,00060_Mean_cd,1013_SWE_cm,1113_SWE_cm,1115_SWE_cm,823_SWE_cm,DAN_SWE_cm,SLI_SWE_cm,TUM_SWE_cm,dayl_s,...,potential_energy,potential_evaporation,pressure,shortwave_radiation,specific_humidity,temperature,total_precipitation,wind_u,wind_v,site_no


In [35]:
Hydro_df.tail()

,00060_Mean,00060_Mean_cd,1013_SWE_cm,1113_SWE_cm,1115_SWE_cm,823_SWE_cm,DAN_SWE_cm,SLI_SWE_cm,TUM_SWE_cm,dayl_s,...,potential_energy,potential_evaporation,pressure,shortwave_radiation,specific_humidity,temperature,total_precipitation,wind_u,wind_v,site_no


## Data Exploration Exercise #1

Use the combined data frame you created for a **USGS NWIS site other than the example** to explore different data and relationships. Select a single water year (October - September) to explore the following relationships:
* Streamflow and SWE (e.g., Daymet and SWE from Snotel)
* Snowmelt -  SWE and temperature/SW radiation
* Rainfall/runoff response - precipitation and streamflow
* comparison of the NLDAS precipitation and temperature with DayMet precipitation and temperature

In markdown below each figure, write a few sentences describing the relationships and anything that is unexpected. This exercise prepares you for HW#2.


In [38]:
# FIX timezone issue FIRST
SNOTEL_df.index = pd.to_datetime(SNOTEL_df.index).tz_localize(None)
PyDayMet_df.index = pd.to_datetime(PyDayMet_df.index).tz_localize(None)
NLDAS_df.index = pd.to_datetime(NLDAS_df.index).tz_localize(None)
streamflow_df.index = pd.to_datetime(streamflow_df.index).tz_localize(None)

# find common date range
begin_date = max([df.index.min() for df in [SNOTEL_df, PyDayMet_df, streamflow_df, NLDAS_df]])
end_date = min([df.index.max() for df in [SNOTEL_df, PyDayMet_df, streamflow_df, NLDAS_df]])

# clip each dataframe
SNOTEL_df = SNOTEL_df[(SNOTEL_df.index >= begin_date) & (SNOTEL_df.index <= end_date)]
PyDayMet_df = PyDayMet_df[(PyDayMet_df.index >= begin_date) & (PyDayMet_df.index <= end_date)]
streamflow_df = streamflow_df[(streamflow_df.index >= begin_date) & (streamflow_df.index <= end_date)]
NLDAS_df = NLDAS_df[(NLDAS_df.index >= begin_date) & (NLDAS_df.index <= end_date)]

# merge
Hydro_df = pd.concat([SNOTEL_df, PyDayMet_df, NLDAS_df, streamflow_df], axis=1)

# move streamflow columns to front
cols = Hydro_df.columns.tolist()
cols = cols[-2:] + cols[:-2]
Hydro_df = Hydro_df[cols]

# fill missing values
Hydro_df = Hydro_df.fillna(0)

Hydro_df.head()

,00060_Mean,00060_Mean_cd,1013_SWE_cm,1113_SWE_cm,1115_SWE_cm,823_SWE_cm,DAN_SWE_cm,SLI_SWE_cm,TUM_SWE_cm,dayl_s,...,potential_energy,potential_evaporation,pressure,shortwave_radiation,specific_humidity,temperature,total_precipitation,wind_u,wind_v,site_no


## Data Exploration Exercise #2

Develop a Python script (.py) that you can run in the terminal that collects, processes, and saves your dataframes and a few select figures of interest.